# <center>**Chapter 17 - Training Loops and Learning Rate Schedulers**<center>

Notebook dùng dữ liệu Ionosphere để minh họa lịch learning rate trong vòng lặp huấn luyện.

In [ ]:
import csv
from pathlib import Path

import torch
from torch import nn
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, LambdaLR
from sklearn.model_selection import train_test_split

def find_data_file(filename):
    start = Path.cwd().resolve()
    patterns = (f"data/**/{filename}", f"*/data/**/{filename}", f"*/*/data/**/{filename}")
    for root in (start, *start.parents):
        for pattern in patterns:
            matches = list(root.glob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Cannot find {filename} near the current directory")

In [2]:
with find_data_file("ionosphere.data").open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.reader(file))
X = torch.tensor([[float(value) for value in row[:34]] for row in rows], dtype=torch.float32)
# Mã hóa nhãn: lớp b thành 0, lớp g thành 1.
y = torch.tensor([[1.0 if row[34] == "g" else 0.0] for row in rows], dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, shuffle=True, random_state=42, stratify=y
)

In [3]:
def build_model():
    return nn.Sequential(nn.Linear(34, 34), nn.ReLU(), nn.Linear(34, 1), nn.Sigmoid())

def run_training(use_schedule, epochs=50):
    torch.manual_seed(42)
    model = build_model()
    optimizer = SGD(model.parameters(), lr=0.1)
    scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.5, total_iters=30) if use_schedule else None
    loss_fn = nn.BCELoss()
    learning_rates = []
    batch_size = 24
    for epoch in range(epochs):
        model.train()
        for start in range(0, len(X_train), batch_size):
            features = X_train[start:start + batch_size]
            targets = y_train[start:start + batch_size]
            loss = loss_fn(model(features), targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        learning_rates.append(optimizer.param_groups[0]["lr"])
        if scheduler is not None:
            scheduler.step()
    model.eval()
    with torch.no_grad():
        accuracy = ((model(X_test) >= 0.5) == y_test.bool()).float().mean().item()
    return accuracy, learning_rates

In [4]:
fixed_accuracy, fixed_lrs = run_training(use_schedule=False)
scheduled_accuracy, scheduled_lrs = run_training(use_schedule=True)
print(f"Fixed LR: start={fixed_lrs[0]:.4f}, end={fixed_lrs[-1]:.4f}, accuracy={fixed_accuracy:.1%}")
print(f"LinearLR: start={scheduled_lrs[0]:.4f}, end={scheduled_lrs[-1]:.4f}, accuracy={scheduled_accuracy:.1%}")

Fixed LR: start=0.1000, end=0.1000, accuracy=92.5%
LinearLR: start=0.1000, end=0.0500, accuracy=93.4%


In [ ]:
# Ví dụ LambdaLR: cập nhật trọng số trước mỗi lần gọi scheduler.
probe_model = nn.Linear(1, 1)
probe_optimizer = SGD(probe_model.parameters(), lr=0.1)
custom_scheduler = LambdaLR(probe_optimizer, lr_lambda=lambda epoch: 0.5 ** (epoch // 10))
probe_input = torch.ones((1, 1), dtype=torch.float32)
custom_rates = [probe_optimizer.param_groups[0]["lr"]]
for _ in range(20):
    probe_optimizer.zero_grad()
    probe_model(probe_input).sum().backward()
    probe_optimizer.step()
    custom_scheduler.step()
    custom_rates.append(probe_optimizer.param_groups[0]["lr"])
print("LambdaLR sample rates:", [round(rate, 4) for rate in custom_rates[::5]])

LambdaLR sample rates: [0.1, 0.1, 0.05, 0.05, 0.025]


Theo ví dụ Ionosphere: phân loại nhị phân bằng MLP 34 → 34 → 1, BCE và SGD. Notebook so sánh learning rate cố định với LinearLR giảm từ 0.1 xuống 0.05 trong 30 epoch, rồi minh họa LambdaLR tùy chỉnh.

# <center>**HẾT**<center>